# Lab 21 — lượt chạy nộp bài (fork của tôi, T4)

Runtime > Change runtime type > **T4 GPU**. Chạy từ trên xuống. **Không** đặt `EVAL_LIMIT` (lượt nộp bài dùng tập eval đầy đủ).

| Ô | Việc | Thời gian ước tính |
|---|---|---|
| 1 | clone fork + cài + gắn Drive | ~2 ph |
| 2 | smoke test | ~30 s |
| 3 | NB1 + **chọn prompt (b) trên val split** + áp prompt | ~8 ph |
| 4 | NB2 (đóng băng mốc) → NB3 → NB4 → NB5 | ~100–130 ph |
| 5 | thí nghiệm replay + NB6 (thưởng) | ~35 ph |
| 6 | verify + nén kết quả để tải về | ~1 ph |

Sau mỗi ô, `results/` và `adapters/` được chép sang Drive (`MyDrive/lab21_backup`). Nếu bị ngắt: chạy lại ô 1 (nó tự khôi phục từ Drive), rồi chạy tiếp ô đang dở — NB4 tự bỏ qua adapter đã train.

In [ ]:
# @title 1. Setup — clone fork + install + Drive backup
import os, subprocess, sys, shutil
REPO = "https://github.com/hungviet1803-lgtm/Day21-Track3-Finetuning-Lab.git"
os.chdir("/content")
if not os.path.exists("Day21-Track3-Finetuning-Lab"):
    subprocess.run(["git", "clone", "-q", REPO], check=True)
os.chdir("/content/Day21-Track3-Finetuning-Lab")
subprocess.run(["git", "pull", "-q"], check=False)
sys.path.insert(0, "src")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)

from google.colab import drive
drive.mount("/content/drive")
BACKUP = "/content/drive/MyDrive/lab21_backup"
os.makedirs(BACKUP, exist_ok=True)

def backup():
    for d in ("results", "adapters", "data/split"):
        if os.path.exists(d):
            shutil.copytree(d, f"{BACKUP}/{d}", dirs_exist_ok=True)
    if os.path.exists("src/labkit/config.py"):
        shutil.copy("src/labkit/config.py", f"{BACKUP}/config.py")
    print("backed up ->", BACKUP)

# Resume after a disconnect: restore whatever the previous session finished.
for d in ("results", "adapters", "data/split"):
    if os.path.exists(f"{BACKUP}/{d}"):
        shutil.copytree(f"{BACKUP}/{d}", d, dirs_exist_ok=True)
        print("restored", d)
if os.path.exists(f"{BACKUP}/config.py"):
    shutil.copy(f"{BACKUP}/config.py", "src/labkit/config.py"); print("restored config.py")

os.environ["COMPUTE_TIER"] = "T4"
# NB1 measured p95=98 tokens (max 101; worst-case replay row 219) -> 256, the next
# power of two. Nothing is truncated; the tier's 1024 is only a VRAM ceiling.
os.environ["MAX_LENGTH"] = "256"
os.environ.pop("EVAL_LIMIT", None)
import torch
print("commit :", subprocess.run(["git","rev-parse","--short","HEAD"], capture_output=True, text=True).stdout.strip())
print("GPU    :", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE — chọn T4 GPU!")


In [ ]:
# @title 2. Smoke — imports, data, unit tests
!python scripts/verify.py --smoke


In [ ]:
# @title 3. NB1 + chọn prompt (b) trên val split (KHÔNG dùng eval_target) + áp prompt
!python -u notebooks/01_data_and_mask.py
!python -u scripts/prompt_dev.py
!python scripts/apply_prompt.py
!git diff --stat src/labkit/config.py
backup()


In [ ]:
# @title 4. Core: NB2 (đóng băng mốc) → NB3 → NB4 → NB5
# Chạy lại ô này sau khi bị ngắt: đổi STAGES thành các stage còn thiếu.
STAGES = "nb2 nb3 nb4 nb5"   # @param {type:"string"}
os.environ.pop("EVAL_LIMIT", None)
for st in STAGES.split():
    rc = subprocess.run([sys.executable, "scripts/colab_run.py", st]).returncode
    backup()
    if rc != 0:
        raise SystemExit(f"{st} failed (exit {rc})")


In [ ]:
# @title 5. Thí nghiệm replay (1 biến: +5% dữ liệu phổ thông) + NB6 merge/hot-swap
!python -u scripts/replay_experiment.py
backup()
!python -u scripts/colab_run.py nb6
backup()


In [ ]:
# @title 6. Gatekeeper + nén kết quả để tải về
!python scripts/verify.py
!cat results/runs.csv
!cat results/verdict.json
!zip -qr /content/lab21_results.zip results adapters/correct/adapter_model.safetensors adapters/correct/adapter_config.json src/labkit/config.py data/replay_selfgen.jsonl
shutil.copy("/content/lab21_results.zip", BACKUP)
from google.colab import files
files.download("/content/lab21_results.zip")
